# Aula 4, Predição de evasão

Notebook executável que acompanha a aula [04-predicao-de-evasao.md](../../lessons/modulo-12-learning-analytics/04-predicao-de-evasao.md).

## O que vamos fazer aqui

Vamos treinar um classificador de evasão (regressão logística do Módulo 2) com as métricas dos
alunos, e usá-lo para sinalizar quem está em risco. numpy.

## Treinando o classificador de evasão

Baixo engajamento e baixa acurácia aumentam o risco. Normalizamos as características antes de treinar,
o que é essencial para a estabilidade.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
n = 300

dias = rng.uniform(0, 1, n)
exercicios = rng.uniform(0, 1, n)
engajamento = 0.6 * dias + 0.4 * exercicios + rng.normal(0, 0.05, n)
acuracia = rng.uniform(0, 1, n)
X = np.column_stack([dias, exercicios, engajamento, acuracia])

risco = -3 * engajamento - 1.5 * acuracia + 2 + rng.normal(0, 0.3, n)
evadiu = (risco > 0).astype(int)


def sigmoide(z):
    return 1 / (1 + np.exp(-z))


Xn = (X - X.mean(0)) / X.std(0)
w = np.zeros(Xn.shape[1]); b = 0.0
for _ in range(3000):
    erro = sigmoide(Xn @ w + b) - evadiu
    w -= 0.1 * Xn.T @ erro / n
    b -= 0.1 * erro.mean()

prob = sigmoide(Xn @ w + b)
acuracia_modelo = ((prob >= 0.5).astype(int) == evadiu).mean()
print(f"Evadiram: {evadiu.sum()} de {n}")
print(f"Acurácia do classificador: {acuracia_modelo:.2%}")
print(f"Alunos sinalizados em risco (prob >= 0,7): {(prob >= 0.7).sum()}")

O classificador atinge ~92% de acurácia, bem acima do acaso, e sinaliza os alunos em risco
pelo engajamento e desempenho. Esses são os alunos para os quais o professor deve dirigir a atenção
primeiro, lembrando que a decisão de como ajudar continua humana.

O projeto que fecha o módulo é o dashboard completo, em `projects/m12-analytics-dashboard/`, que reúne
métricas, engajamento e risco de evasão de uma turma.